In [ ]:

import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.widgets import Slider
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.preprocessing import StandardScaler

In [ ]:
# ── Column name constants ──────────────────────────────────────────────────────
OUTPUT_COLS  = ['y_night', 'y_peak', 'UTCI']
DROP_COLS    = ['alpha']         # mirroring X.alpha = [] in the MATLAB code
YLIM         = (12, 37)          # fixed y-axis range matching MATLAB

# Display names shown as subplot titles (order must match column order after drop)
PARAM_DISPLAY = {
    'albedo_w' : 'albedo_w',
    'albedo_r' : 'albedo_r',
    'height_c' : 'height_c',
    'width_c'  : 'width_c',
    'width_r'  : 'width_r',
    'fveg_g'   : 'fveg_g',
    'lan_g'    : 'lan_g',
    'cv_g'     : 'cv_g',
    'lan_w'    : 'lan_w',
    'cv_w'     : 'cv_w',
    'radius_t' : 'radius_t',
}

# Tick positions per parameter (same as MATLAB xtickVals)
PARAM_TICKS = {
    'albedo_w' : [0.4,  0.6,  0.8 ],
    'albedo_r' : [0.2,  0.5,  0.8 ],
    'height_c' : [16,   20,   24  ],
    'width_c'  : [16,   22,   28  ],
    'width_r'  : [10,   16,   22  ],
    'fveg_g'   : [0.1,  0.5,  0.9 ],
    'lan_g'    : [0.5,  1.0,  1.5 ],
    'cv_g'     : [1e6,  1.5e6, 2e6],
    'lan_w'    : [0.5,  1.0,  1.5 ],
    'cv_w'     : [1e6,  1.5e6, 2e6],
    'radius_t' : [2,    4,    6   ],
}

COLORS = {
    'y_peak'  : ('#d62728', 'Day'),
    'y_night' : ('#1f77b4', 'Night'),
    'UTCI'    : ('#2ca02c', 'UTCI'),
}

In [ ]:

# ── GP training ────────────────────────────────────────────────────────────────

def build_gp():
    """ARD squared-exponential kernel + white noise, mirroring MATLAB ardsquaredexponential."""
    kernel = RBF(length_scale=np.ones(1), length_scale_bounds=(1e-3, 1e3)) + \
             WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))
    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,      # mirrors 'Standardize',true in MATLAB
        n_restarts_optimizer=3,
        random_state=42,
    )


def train_models(X_train: pd.DataFrame, y_train: dict[str, np.ndarray]):
    """Train one GP per output. Returns dict of fitted models and scalers."""
    models, scalers = {}, {}
    for key, y in y_train.items():
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train)
        gp = build_gp()
        gp.fit(X_scaled, y)
        models[key]  = gp
        scalers[key] = scaler
        print(f"  Trained GP for {key:8s}  |  kernel: {gp.kernel_}")
    return models, scalers

In [ ]:
# ── Prediction helper ──────────────────────────────────────────────────────────

def predict_cut(model, scaler, X_col: np.ndarray, fixed: np.ndarray,
                col_idx: int, n_pts: int = 100):
    """
    Build query matrix: all columns fixed at `fixed`, vary column `col_idx`
    over the range of X_col.  Returns (xi, y_mean, y_lower, y_upper).
    """
    xi = np.linspace(X_col.min(), X_col.max(), n_pts)
    Xq = np.tile(fixed, (n_pts, 1))
    Xq[:, col_idx] = xi
    Xq_scaled = scaler.transform(Xq)
    y_mean, y_std = model.predict(Xq_scaled, return_std=True)
    return xi, y_mean, y_mean - 1.96 * y_std, y_mean + 1.96 * y_std



In [ ]:
# ── Main visualization ─────────────────────────────────────────────────────────

def run(csv_path: str):
    # ── Load data ──────────────────────────────────────────────────────────────
    df = pd.read_csv(csv_path)

    # Separate inputs / outputs
    y_df = df[OUTPUT_COLS].copy()
    X_df = df.drop(columns=OUTPUT_COLS + [c for c in DROP_COLS if c in df.columns])

    param_names = list(X_df.columns)
    n_params    = len(param_names)
    n_samples   = len(X_df)

    print(f"Loaded {n_samples} samples, {n_params} input parameters, 3 outputs.")

    # ── Train / test split (85 / 15, random) ──────────────────────────────────
    rng      = np.random.default_rng(42)
    idx      = rng.permutation(n_samples)
    n_train  = round(0.85 * n_samples)
    train_i, test_i = idx[:n_train], idx[n_train:]

    X_train  = X_df.iloc[train_i]
    y_train  = {k: y_df[k].iloc[train_i].values for k in OUTPUT_COLS}

    print("Training GP models …")
    models, scalers = train_models(X_train, y_train)

    # Fixed point initialised at column means (mirrors varfun(@mean, X) in MATLAB)
    fixed = X_df.mean().values.copy()   # shape (n_params,)

    # ── Build figure ───────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(26, 6), constrained_layout=False)
    fig.suptitle('Interactive GP Cuts with Sliders', fontsize=11)

    # Reserve bottom area for sliders
    slider_total_h = 0.10
    top_h          = 1.0 - slider_total_h - 0.12   # 0.12 for suptitle / margins

    gs_plots   = gridspec.GridSpec(
        1, n_params, figure=fig,
        left=0.04, right=0.99,
        top=0.88, bottom=slider_total_h + 0.08,
        wspace=0.35,
    )
    gs_sliders = gridspec.GridSpec(
        1, n_params, figure=fig,
        left=0.04, right=0.99,
        top=slider_total_h, bottom=0.01,
        wspace=0.35,
    )

    axes    = [fig.add_subplot(gs_plots[0, i]) for i in range(n_params)]
    s_axes  = [fig.add_subplot(gs_sliders[0, i]) for i in range(n_params)]

    # ── Plot initial GP cuts ───────────────────────────────────────────────────
    lines   = {k: [] for k in OUTPUT_COLS}
    fills   = {k: [] for k in OUTPUT_COLS}
    vlines  = []

    for i, (ax, pname) in enumerate(zip(axes, param_names)):
        X_col = X_df[pname].values

        for key in OUTPUT_COLS:
            xi, ym, yl, yu = predict_cut(models[key], scalers[key], X_col, fixed, i)
            c, lbl = COLORS[key]

            poly = ax.fill_between(xi, yl, yu, color=c, alpha=0.15, linewidth=0)
            line, = ax.plot(xi, ym, color=c, linewidth=1.2,
                            label=lbl if i == 0 else '_nolegend_')

            lines[key].append(line)
            fills[key].append(poly)

        # Vertical dashed line at initial (midpoint) fixed value
        xmid = 0.5 * (X_col.min() + X_col.max())
        vl   = ax.axvline(xmid, color='k', linestyle='--', linewidth=1.0)
        vlines.append(vl)

        ax.set_xlim(X_col.min(), X_col.max())
        ax.set_ylim(YLIM)
        ax.set_title(PARAM_DISPLAY.get(pname, pname), fontsize=9)
        ax.grid(True, linewidth=0.4, alpha=0.6)
        ax.tick_params(labelsize=7)

        ticks = PARAM_TICKS.get(pname)
        if ticks:
            ax.set_xticks(ticks)

        if i == 0:
            ax.set_ylabel('Temperature / UTCI [°C]', fontsize=8)
        else:
            ax.set_yticklabels([])

    # Shared legend below plots
    handles = [lines[k][0] for k in OUTPUT_COLS]
    labels  = [COLORS[k][1] for k in OUTPUT_COLS]
    fig.legend(handles, labels, loc='upper center',
               bbox_to_anchor=(0.5, 0.94), ncol=3, fontsize=9, frameon=False)

    # ── Sliders ────────────────────────────────────────────────────────────────
    sliders = []
    for i, (sax, pname) in enumerate(zip(s_axes, param_names)):
        X_col = X_df[pname].values
        sl = Slider(
            ax=sax,
            label='',
            valmin=X_col.min(),
            valmax=X_col.max(),
            valinit=fixed[i],
            color='steelblue',
        )
        sl.label.set_fontsize(6)
        sl.valtext.set_fontsize(6)
        sliders.append(sl)

    # ── Callback ───────────────────────────────────────────────────────────────
    def update(val, dim_changed=None):
        # Update the fixed point from all slider values
        for j, sl in enumerate(sliders):
            fixed[j] = sl.val

        for j, (ax, pname) in enumerate(zip(axes, param_names)):
            X_col = X_df[pname].values

            for key in OUTPUT_COLS:
                xi, ym, yl, yu = predict_cut(models[key], scalers[key], X_col, fixed, j)
                lines[key][j].set_ydata(ym)

                # Update fill_between polygon (remove old, draw new in same axis)
                fills[key][j].remove()
                c = COLORS[key][0]
                fills[key][j] = ax.fill_between(xi, yl, yu, color=c, alpha=0.15, linewidth=0)

            # Move vertical dashed line to current fixed value
            vlines[j].set_xdata([fixed[j], fixed[j]])

        fig.canvas.draw_idle()

    for sl in sliders:
        sl.on_changed(update)

    plt.show()

    # Snapshot export (vector PDF, mirrors exportgraphics in MATLAB)
    fig.savefig('GP_slider_snapshot.pdf', format='pdf', bbox_inches='tight')
    print("Saved GP_slider_snapshot.pdf")


# ── CLI entry point ────────────────────────────────────────────────────────────
if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Interactive GP slider plot')
    parser.add_argument('--csv', default='InputOutput.csv',
                        help='Path to InputOutput.csv (default: InputOutput.csv)')
    args = parser.parse_args()
    run(args.csv)
